# 04. Publish One Cell Type To Battery Genome

This notebook shows the intended human-facing flow:

- define one `CellType` in Python
- call `publish(cell_type, destination=...)`
- get back the canonical BattINFO record path and, optionally, the Battery Genome page URL

The canonical JSON record and registry submission package are still generated, but they stay behind the main BattINFO publish surface.

In [ ]:
import json
import os
from pathlib import Path

from battinfo import CellType, publish, quantity, source


## 1. Define the cell type in Python

This is the only semantic authoring step. No hand-edited JSON is required.

In [ ]:
workspace_root = Path('.battinfo/notebooks/04-cell-type-publish')

cell_type = CellType(
    manufacturer='Google',
    model='G20M7',
    name='Google G20M7',
    format='prismatic',
    chemistry='Li-ion',
    size_code='P6/65/75',
    iec_code='ICP6/65/75',
    country_of_origin='Vietnam',
    year=2025,
    positive_electrode_basis='LCO',
    negative_electrode_basis='graphite',
    nominal_voltage=quantity(3.9, 'V'),
    rated_energy=quantity(19.39, 'Wh'),
    source=source(type='label', file='google-g20m7-2025.json'),
)

{
    'manufacturer': cell_type.manufacturer,
    'model': cell_type.model,
    'format': cell_type.format,
    'chemistry': cell_type.chemistry,
}


## 2. Publish locally

This writes the canonical BattINFO record to disk. For humans, this is the simplest way to see the generated source-of-truth artifact.

In [ ]:
local_result = publish(
    cell_type,
    destination='local',
    root=workspace_root,
    force=True,
)

record_path = Path(local_result.debug_paths['canonical_record_path'])
record = json.loads(record_path.read_text(encoding='utf-8'))

{
    'canonical_id': local_result.canonical_id,
    'canonical_iri': local_result.canonical_iri,
    'record_path': str(record_path),
    'saved_manufacturer': record['product']['manufacturer']['name'],
    'saved_model': record['product']['model'],
}


## 3. Optionally publish to a registry or Battery Genome

Set these environment variables before running the next cell:

- `BATTINFO_DEMO_REGISTRY_URL`
- `BATTINFO_DEMO_API_KEY`
- optional: `BATTERY_GENOME_PLATFORM_URL`

If `BATTERY_GENOME_PLATFORM_URL` is set, BattINFO publishes with `destination='battery-genome'` and returns the expected page URL. Otherwise it publishes only to the registry.

In [ ]:
registry_url = os.environ.get('BATTINFO_DEMO_REGISTRY_URL')
api_key = os.environ.get('BATTINFO_DEMO_API_KEY')
platform_url = os.environ.get('BATTERY_GENOME_PLATFORM_URL')
remote_root = workspace_root.parent / '04-cell-type-publish-remote'

if not registry_url or not api_key:
    remote_result = {
        'status': 'skipped',
        'reason': 'Set BATTINFO_DEMO_REGISTRY_URL and BATTINFO_DEMO_API_KEY to run the remote publish step.',
    }
else:
    destination = 'battery-genome' if platform_url else 'registry'
    remote_publish = publish(
        cell_type,
        destination=destination,
        root=remote_root,
        force=True,
        registry_base_url=registry_url,
        api_key=api_key,
        platform_base_url=platform_url,
        workspace_id='hello-world',
        publisher_id='demo-lab',
        source_version='demo-2026-03-24',
    )
    remote_result = remote_publish.model_dump(mode='json')

remote_result
